In [0]:
%python
# ============================================================================
# IMPORT PROFESSIONAL EXTRACTION UTILITIES FOR GOLD ENRICHMENT
# ============================================================================

import sys

# Add project paths to sys.path
project_root = "/Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import professional utilities for gold table enrichment
HAS_SPECIALTY_MODELS = False
HAS_STRUCTURED_VALIDATION = False

try:
    from prompts_and_pydantic_models.medical_specialties import MedicalSpecialties
    print("✓ Imported: Medical specialties models")
    HAS_SPECIALTY_MODELS = True
except ImportError as e:
    print(f"⚠️  Medical specialties models unavailable: {str(e)[:60]}...")

try:
    from prompts_and_pydantic_models.facility_and_ngo_fields import Facility, NGO
    print("✓ Imported: Structured Facility/NGO models for validation")
    HAS_STRUCTURED_VALIDATION = True
except ImportError as e:
    print(f"⚠️  Structured models unavailable: {str(e)[:60]}...")

if HAS_SPECIALTY_MODELS or HAS_STRUCTURED_VALIDATION:
    print("\n✅ Professional utilities available for gold enrichment")
    print("   → Medical specialty classification")
    print("   → Structured field validation")
else:
    print("\n⚠️  Professional utilities not available - using standard gold build")

In [0]:
%python
# Databricks notebook source
from pyspark.sql import functions as F

CATALOG = "vf_health"
SCHEMA = "ghana"

SILVER = f"{CATALOG}.{SCHEMA}.silver_facilities_clean"
GOLD_PROFILES = f"{CATALOG}.{SCHEMA}.gold_facility_profiles"
GOLD_CLAIMS = f"{CATALOG}.{SCHEMA}.gold_facility_claims"
GOLD_CITATIONS = f"{CATALOG}.{SCHEMA}.gold_citations"
GOLD_RISK = f"{CATALOG}.{SCHEMA}.gold_risk_signals"
GOLD_DESERTS = f"{CATALOG}.{SCHEMA}.gold_medical_deserts"

df = spark.table(SILVER)

print("Building Gold Tables with Intelligence Layer...")
print("="*70)

# ============================================================================
# 1) GOLD PROFILES - WITH COMPUTED INTELLIGENCE FIELDS
# ============================================================================
print("\n1️⃣  Building gold_facility_profiles with computed fields...")

profiles = (
    df
    # Detect ICU capability
    .withColumn(
        "has_ICU",
        F.expr(
            "exists(capability, x -> lower(x) like '%icu%' or lower(x) like '%intensive%' or lower(x) like '%critical care%') "
            "OR exists(equipment, x -> lower(x) like '%ventilator%' or lower(x) like '%icu%')"
        )
    )
    
    # Detect surgery capability
    .withColumn(
        "has_surgery",
        F.expr(
            "exists(procedure, x -> lower(x) like '%surgery%' or lower(x) like '%surgical%' or lower(x) like '%operation%') "
            "OR exists(capability, x -> lower(x) like '%surgery%' or lower(x) like '%surgical%')"
        )
    )
    
    # Detect emergency readiness
    .withColumn(
        "emergency_ready",
        F.expr(
            "exists(capability, x -> lower(x) like '%emergency%' or lower(x) like '%24%' or lower(x) like '%trauma%') "
            "OR exists(specialties, x -> lower(x) like '%emergency%')"
        )
    )
    
    # Calculate doctor-to-capacity ratio
    .withColumn(
        "doctor_ratio",
        F.when(
            (F.col("capacity").isNotNull()) & (F.col("capacity") > 0) & 
            (F.col("numberDoctors").isNotNull()) & (F.col("numberDoctors") > 0),
            F.round(F.col("numberDoctors") / F.col("capacity"), 3)
        ).otherwise(F.lit(None))
    )
    
    # Calculate facility score (0-5 scale)
    .withColumn(
        "facility_score",
        (
            # Base score
            F.lit(1.0) +
            # +1 for ICU
            F.when(F.col("has_ICU"), F.lit(1.0)).otherwise(F.lit(0.0)) +
            # +1 for surgery
            F.when(F.col("has_surgery"), F.lit(1.0)).otherwise(F.lit(0.0)) +
            # +1 for emergency readiness
            F.when(F.col("emergency_ready"), F.lit(1.0)).otherwise(F.lit(0.0)) +
            # +0.5 for good doctor ratio (>= 0.02)
            F.when((F.col("doctor_ratio").isNotNull()) & (F.col("doctor_ratio") >= 0.02), F.lit(0.5)).otherwise(F.lit(0.0)) +
            # +0.5 for having contact info
            F.when(
                (F.size(F.col("phone_numbers")) > 0) | F.col("email").isNotNull() | (F.size(F.col("websites")) > 0),
                F.lit(0.5)
            ).otherwise(F.lit(0.0))
        )
    )
    
    # Create profile JSON
    .withColumn("profile_json", F.to_json(F.struct(*[F.col(c) for c in df.columns if c != "row_id"])))
    .select(
        "row_id", "unique_id", "name", "organization_type", "profile_json",
        "has_ICU", "has_surgery", "emergency_ready", "doctor_ratio", "facility_score"
    )
    .withColumn("updated_at", F.current_timestamp())
)
profiles.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(GOLD_PROFILES)
print(f"   ✓ Created {spark.table(GOLD_PROFILES).count()} facility profiles with intelligence")

# ============================================================================
# 2) GOLD CLAIMS - WITH CONFIDENCE SCORING
# ============================================================================
print("\n2️⃣  Building gold_facility_claims with validation-based confidence...")

# Procedure claims with validation
proc_claims = (
    df.select("row_id", "unique_id", F.explode_outer("procedure").alias("claim_text"))
      .filter(F.col("claim_text").isNotNull() & (F.trim("claim_text") != "") & (F.trim("claim_text") != '""'))
      .withColumn("claim_type", F.lit("procedure"))
      # Higher confidence for standardized terms
      .withColumn(
          "confidence",
          F.when(
              F.lower(F.col("claim_text")).rlike("surgery|surgical|cesarean|immunization|childbirth"),
              F.lit(0.90)
          ).otherwise(F.lit(0.75))
      )
)

# Equipment claims with validation
equip_claims = (
    df.select("row_id", "unique_id", F.explode_outer("equipment").alias("claim_text"))
      .filter(F.col("claim_text").isNotNull() & (F.trim("claim_text") != "") & (F.trim("claim_text") != '""'))
      .withColumn("claim_type", F.lit("equipment"))
      # Higher confidence for standardized equipment names
      .withColumn(
          "confidence",
          F.when(
              F.lower(F.col("claim_text")).rlike("x-ray|mri|ct_scanner|ultrasound|ecg"),
              F.lit(0.95)
          ).otherwise(F.lit(0.80))
      )
)

# Capability claims with validation
cap_claims = (
    df.select("row_id", "unique_id", F.explode_outer("capability").alias("claim_text"))
      .filter(F.col("claim_text").isNotNull() & (F.trim("claim_text") != "") & (F.trim("claim_text") != '""'))
      .withColumn("claim_type", F.lit("capability"))
      # Higher confidence for standardized capabilities
      .withColumn(
          "confidence",
          F.when(
              F.lower(F.col("claim_text")).rlike("emergency_care|icu_services|24_hour_service|maternal_care"),
              F.lit(0.90)
          ).otherwise(F.lit(0.70))
      )
)

claims = (
    proc_claims.unionByName(equip_claims).unionByName(cap_claims)
    .withColumn("claim_id", F.sha2(F.concat_ws("||", "row_id", "claim_type", "claim_text"), 256))
    .withColumn("created_at", F.current_timestamp())
    .select("claim_id", "row_id", "unique_id", "claim_type", "claim_text", "confidence", "created_at")
)
claims.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(GOLD_CLAIMS)
print(f"   ✓ Created {spark.table(GOLD_CLAIMS).count()} validated claims (IDP output)")

# ============================================================================
# 3) GOLD CITATIONS - TRACEABILITY
# ============================================================================
print("\n3️⃣  Building gold_citations for data lineage...")

citations = (
    df.select("row_id", "source_url", "procedure", "equipment", "capability")
      .withColumn(
          "evidence_text",
          F.concat_ws(
              " | ",
              F.coalesce(F.array_join("procedure", "; "), F.lit("")),
              F.coalesce(F.array_join("equipment", "; "), F.lit("")),
              F.coalesce(F.array_join("capability", "; "), F.lit(""))
          )
      )
      .withColumn("field", F.lit("procedure/equipment/capability"))
      .withColumn("step_id", F.lit("gold_build_v2_standardized"))
      .withColumn("citation_id", F.sha2(F.concat_ws("||", "row_id", "source_url", "field"), 256))
      .withColumn("created_at", F.current_timestamp())
      .select("citation_id", "row_id", "source_url", "field", "evidence_text", "step_id", "created_at")
)
citations.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(GOLD_CITATIONS)
print(f"   ✓ Created {spark.table(GOLD_CITATIONS).count()} citations for traceability")

# ============================================================================
# 4) GOLD RISK SIGNALS - ENHANCED INTELLIGENCE LAYER 🔥
# ============================================================================
print("\n4️⃣  Building gold_risk_signals with critical detection rules...")

# Load profiles for risk detection
profiles_df = spark.table(GOLD_PROFILES)

risk = (
    profiles_df
    .join(df.select("row_id", "capacity", "numberDoctors", "phone_numbers", "email", "websites", 
                    "capability", "specialties", "equipment"), "row_id")
    
    # RULE 1: Missing contact info
    .withColumn("has_no_contact", 
        (F.size("phone_numbers") == 0) & F.col("email").isNull() & (F.size("websites") == 0)
    )
    
    # RULE 2: Bad capacity (negative values - should be caught in silver but double-check)
    .withColumn("bad_capacity", 
        F.col("capacity").isNotNull() & (F.col("capacity") < 0)
    )
    
    # RULE 3: 🚨 CRITICAL - Surgery without ICU
    .withColumn("surgery_no_icu",
        F.col("has_surgery") & ~F.col("has_ICU")
    )
    
    # RULE 4: ⚠️ SUSPICIOUS - Cardiac surgery claim without ICU equipment
    .withColumn("suspicious_cardiac",
        F.expr(
            "exists(capability, x -> lower(x) like '%cardiac%' or lower(x) like '%heart surgery%') "
            "AND NOT exists(equipment, x -> lower(x) like '%icu%' or lower(x) like '%ventilator%')"
        )
    )
    
    # RULE 5: ⚠️ STAFF SHORTAGE - High capacity, low doctors
    .withColumn("staff_shortage",
        (F.col("capacity").isNotNull()) & (F.col("capacity") > 50) &
        (F.col("numberDoctors").isNotNull()) & (F.col("numberDoctors") < 3)
    )
    
    # RULE 6: ⚠️ HIGH LOAD - Doctor ratio too low (< 0.01 = 1 doctor per 100+ beds)
    .withColumn("high_patient_load",
        (F.col("doctor_ratio").isNotNull()) & (F.col("doctor_ratio") < 0.01)
    )
    
    # Determine signal type and score
    .withColumn(
        "signal_type",
        F.when(F.col("surgery_no_icu"), F.lit("critical_surgery_no_icu"))
         .when(F.col("suspicious_cardiac"), F.lit("suspicious_cardiac_claim"))
         .when(F.col("staff_shortage"), F.lit("warning_staff_shortage"))
         .when(F.col("high_patient_load"), F.lit("warning_high_patient_load"))
         .when(F.col("bad_capacity"), F.lit("anomaly_capacity"))
         .when(F.col("has_no_contact"), F.lit("missing_contact"))
         .otherwise(F.lit("none"))
    )
    .withColumn(
        "signal_score",
        F.when(F.col("surgery_no_icu"), F.lit(0.95))  # Critical
         .when(F.col("suspicious_cardiac"), F.lit(0.85))  # High risk
         .when(F.col("staff_shortage"), F.lit(0.80))  # High risk
         .when(F.col("high_patient_load"), F.lit(0.75))  # Medium risk
         .when(F.col("bad_capacity"), F.lit(0.70))  # Data quality
         .when(F.col("has_no_contact"), F.lit(0.60))  # Data quality
         .otherwise(F.lit(0.0))
    )
    .withColumn(
        "signal_flag",
        F.col("signal_type") != "none"
    )
    .withColumn(
        "reason",
        F.when(F.col("surgery_no_icu"), 
               F.lit("🚨 CRITICAL: Offers surgery but lacks ICU capability"))
         .when(F.col("suspicious_cardiac"), 
               F.lit("⚠️ SUSPICIOUS: Claims cardiac surgery without ICU equipment"))
         .when(F.col("staff_shortage"), 
               F.concat(F.lit("⚠️ STAFF SHORTAGE: "), F.col("numberDoctors"), F.lit(" doctors for "), F.col("capacity"), F.lit(" beds")))
         .when(F.col("high_patient_load"), 
               F.concat(F.lit("⚠️ HIGH LOAD: Doctor ratio "), F.col("doctor_ratio")))
         .when(F.col("bad_capacity"), F.lit("capacity < 0"))
         .when(F.col("has_no_contact"), F.lit("no phone/email/website"))
         .otherwise(F.lit("ok"))
    )
    .withColumn("signal_id", F.sha2(F.concat_ws("||", "row_id", "signal_type", "reason"), 256))
    .withColumn("created_at", F.current_timestamp())
    .select("signal_id", "row_id", "unique_id", "signal_type", "signal_score", "signal_flag", "reason", "created_at")
)
risk.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(GOLD_RISK)
print(f"   ✓ Created {spark.table(GOLD_RISK).count()} risk signals")

# Show critical risks
critical_risks = spark.table(GOLD_RISK).filter(F.col("signal_score") >= 0.80).count()
print(f"   🚨 {critical_risks} CRITICAL/HIGH RISK signals detected")

# ============================================================================
# 5) GOLD MEDICAL DESERTS - ENHANCED REGIONAL INTELLIGENCE
# ============================================================================
print("\n5️⃣  Building gold_medical_deserts with enhanced scoring...")

# Join with profiles for computed fields
deserts = (
    profiles_df
    .join(df.select("row_id", "address_stateOrRegion", "address_city", 
                    "capacity", "numberDoctors", "specialties", "capability"), "row_id")
    .withColumn("region", F.coalesce(F.col("address_stateOrRegion"), F.lit("UNKNOWN")))
    .withColumn("district", F.coalesce(F.col("address_city"), F.lit("UNKNOWN")))
    .withColumn(
        "is_maternal",
        F.expr(
            "exists(specialties, x -> lower(x) like '%gynecology%' or lower(x) like '%obstetric%') "
            "OR exists(capability, x -> lower(x) like '%maternity%' or lower(x) like '%maternal%')"
        )
    )
    .groupBy("region", "district")
    .agg(
        F.count("*").alias("facility_count"),
        F.sum(F.when(F.col("has_ICU"), 1).otherwise(0)).alias("icu_sites"),
        F.sum(F.when(F.col("emergency_ready"), 1).otherwise(0)).alias("emergency_sites"),
        F.sum(F.when(F.col("is_maternal"), 1).otherwise(0)).alias("maternal_sites"),
        F.sum(F.when(F.col("has_surgery"), 1).otherwise(0)).alias("surgical_sites"),
        F.sum(F.coalesce(F.col("capacity"), F.lit(0))).alias("total_capacity"),
        F.sum(F.coalesce(F.col("numberDoctors"), F.lit(0))).alias("total_doctors"),
        F.avg(F.col("facility_score")).alias("avg_facility_score")
    )
    # Enhanced desert score (0-5, higher = more desert-like)
    .withColumn(
        "desert_score",
        (
            # Low facility density (0-1 scale)
            F.least(F.lit(1.0), F.lit(5.0)/(F.col("facility_count")+F.lit(1.0))) +
            # Low ICU availability (0-1 scale)
            F.least(F.lit(1.0), F.lit(2.0)/(F.col("icu_sites")+F.lit(1.0))) +
            # Low emergency availability (0-1 scale)
            F.least(F.lit(1.0), F.lit(2.0)/(F.col("emergency_sites")+F.lit(1.0))) +
            # Low maternal care (0-1 scale)
            F.least(F.lit(1.0), F.lit(2.0)/(F.col("maternal_sites")+F.lit(1.0))) +
            # Low surgical capability (0-1 scale)
            F.least(F.lit(1.0), F.lit(2.0)/(F.col("surgical_sites")+F.lit(1.0)))
        )
    )
    # Classify desert severity
    .withColumn(
        "desert_severity",
        F.when(F.col("desert_score") >= 4.0, F.lit("CRITICAL"))
         .when(F.col("desert_score") >= 3.0, F.lit("HIGH"))
         .when(F.col("desert_score") >= 2.0, F.lit("MODERATE"))
         .otherwise(F.lit("LOW"))
    )
    .withColumn("computed_at", F.current_timestamp())
)

deserts.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(GOLD_DESERTS)
print(f"   ✓ Created {spark.table(GOLD_DESERTS).count()} regional desert analyses")

# Show critical deserts
critical_deserts = spark.table(GOLD_DESERTS).filter(F.col("desert_severity") == "CRITICAL").count()
print(f"   🚨 {critical_deserts} CRITICAL medical deserts identified")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*70)
print("GOLD TABLES BUILD COMPLETE")
print("="*70)
print(f"✓ gold_facility_profiles:  {spark.table(GOLD_PROFILES).count()} facilities with intelligence")
print(f"✓ gold_facility_claims:    {spark.table(GOLD_CLAIMS).count()} validated IDP claims")
print(f"✓ gold_citations:          {spark.table(GOLD_CITATIONS).count()} traceability records")
print(f"✓ gold_risk_signals:       {spark.table(GOLD_RISK).count()} risk signals ({critical_risks} critical)")
print(f"✓ gold_medical_deserts:    {spark.table(GOLD_DESERTS).count()} regions ({critical_deserts} critical)")
print("="*70)

In [0]:
%python
# ============================================================================
# GOLD TABLE A: REGIONAL SUMMARY WITH GAP SEVERITY ANALYSIS
# ============================================================================
# Aggregate facility data by region to identify medical deserts

from pyspark.sql import functions as F, Window

GOLD_REGION_SUMMARY = f"{CATALOG}.{SCHEMA}.gold_region_summary"

print("\n" + "="*70)
print("🌍 BUILDING REGIONAL SUMMARY GOLD TABLE")
print("="*70)

silver = spark.table(SILVER)

# Regional aggregations
region_summary = (
    silver
    .groupBy("address_region_clean")
    .agg(
        # Basic counts
        F.count("*").alias("total_facilities"),
        
        # Capability metrics
        F.round(F.avg("capability_score"), 2).alias("avg_capability_score"),
        
        # Medical desert indicators
        F.sum(F.when(F.col("is_medical_desert_risk"), 1).otherwise(0)).alias("medical_desert_count"),
        
        # Specific capability counts
        F.sum(F.when(F.col("has_emergency"), 1).otherwise(0)).alias("facilities_with_emergency"),
        F.sum(F.when(F.col("has_surgery"), 1).otherwise(0)).alias("facilities_with_surgery"),
        F.sum(F.when(F.col("has_imaging"), 1).otherwise(0)).alias("facilities_with_imaging"),
        
        # Resource totals (null-safe)
        F.sum(F.coalesce(F.col("numberDoctors"), F.lit(0))).alias("total_doctors"),
        F.sum(F.coalesce(F.col("capacity"), F.lit(0))).alias("total_beds")
    )
    
    # Computed ratios
    .withColumn(
        "doctors_per_facility",
        F.round(F.col("total_doctors") / F.col("total_facilities"), 2)
    )
    .withColumn(
        "desert_pct",
        F.round((F.col("medical_desert_count") / F.col("total_facilities")) * 100, 1)
    )
    
    # Gap severity classification
    .withColumn(
        "gap_severity",
        F.when(
            (F.col("desert_pct") >= 60) | (F.col("avg_capability_score") < 2),
            F.lit("Critical")
        ).when(
            (F.col("desert_pct") >= 40) | (F.col("avg_capability_score") < 3),
            F.lit("High")
        ).when(
            F.col("desert_pct") >= 20,
            F.lit("Moderate")
        ).otherwise(
            F.lit("Low")
        )
    )
    
    # Add creation timestamp
    .withColumn("computed_at", F.current_timestamp())
    
    # Order by severity
    .orderBy(F.col("desert_pct").desc())
)

# Write to Gold
region_summary.write.mode("overwrite").format("delta").saveAsTable(GOLD_REGION_SUMMARY)

final_region_count = spark.table(GOLD_REGION_SUMMARY).count()

print(f"\n✅ Gold region summary created: {GOLD_REGION_SUMMARY}")
print(f"   Total regions analyzed: {final_region_count}")

# Show critical and high severity regions
print("\n🚨 CRITICAL & HIGH SEVERITY REGIONS:")
spark.table(GOLD_REGION_SUMMARY).filter(
    F.col("gap_severity").isin("Critical", "High")
).select(
    "address_region_clean",
    "total_facilities",
    "avg_capability_score",
    "desert_pct",
    "gap_severity"
).show(10, truncate=False)

print("\n📊 SEVERITY DISTRIBUTION:")
spark.table(GOLD_REGION_SUMMARY).groupBy("gap_severity").count().orderBy(
    F.when(F.col("gap_severity") == "Critical", 1)
    .when(F.col("gap_severity") == "High", 2)
    .when(F.col("gap_severity") == "Moderate", 3)
    .otherwise(4)
).show()

print("="*70)

In [0]:
%python
# ============================================================================
# GOLD TABLE B: FACILITY CARDS WITH RAG TEXT
# ============================================================================
# Create enriched facility records with full-text column for vector search

from pyspark.sql import functions as F

GOLD_FACILITY_CARDS = f"{CATALOG}.{SCHEMA}.gold_facility_cards"

print("\n" + "="*70)
print("🏛️ BUILDING FACILITY CARDS GOLD TABLE WITH RAG TEXT")
print("="*70)

silver = spark.table(SILVER)

# Helper function to safely join arrays
def safe_array_join(col_name, delimiter=", "):
    return F.coalesce(
        F.array_join(F.col(col_name), delimiter),
        F.lit("")
    )

# Create facility cards with comprehensive full-text field
facility_cards = (
    silver
    .select(
        "row_id",
        "unique_id",
        "name",
        "organization_type",
        "facilityTypeId",
        "operatorTypeId",
        "specialties",
        "procedure",
        "equipment",
        "capability",
        "address_city",
        "address_region_clean",
        "address_country",
        "phone_numbers",
        "email",
        "websites",
        "officialWebsite",
        "description",
        "organizationDescription",
        "numberDoctors",
        "capacity",
        "area",
        "yearEstablished",
        "acceptsVolunteers",
        "has_emergency",
        "has_surgery",
        "has_imaging",
        "capability_score",
        "is_medical_desert_risk",
        "source_url"
    )
    
    # Build comprehensive full-text field for RAG
    .withColumn(
        "full_text_for_rag",
        F.concat_ws(
            " | ",
            # Facility identification
            F.coalesce(F.col("name"), F.lit("")),
            
            # Type classification
            F.concat(
                F.lit("Type: "),
                F.coalesce(F.col("facilityTypeId"), F.lit("unknown")),
                F.lit(" ("),
                F.coalesce(F.col("operatorTypeId"), F.lit("unknown")),
                F.lit(")")
            ),
            
            # Location
            F.concat(
                F.lit("Location: "),
                F.coalesce(F.col("address_city"), F.lit("")),
                F.when(F.col("address_city").isNotNull(), F.lit(", ")).otherwise(F.lit("")),
                F.col("address_region_clean"),
                F.lit(", "),
                F.coalesce(F.col("address_country"), F.lit("Ghana"))
            ),
            
            # Medical specialties
            F.concat(
                F.lit("Specialties: "),
                F.when(
                    F.size(F.coalesce(F.col("specialties"), F.array())) > 0,
                    safe_array_join("specialties", ", ")
                ).otherwise(F.lit("None listed"))
            ),
            
            # Procedures
            F.concat(
                F.lit("Procedures: "),
                F.when(
                    F.size(F.coalesce(F.col("procedure"), F.array())) > 0,
                    safe_array_join("procedure", "; ")
                ).otherwise(F.lit("None listed"))
            ),
            
            # Equipment
            F.concat(
                F.lit("Equipment: "),
                F.when(
                    F.size(F.coalesce(F.col("equipment"), F.array())) > 0,
                    safe_array_join("equipment", "; ")
                ).otherwise(F.lit("None listed"))
            ),
            
            # Capabilities
            F.concat(
                F.lit("Capabilities: "),
                F.when(
                    F.size(F.coalesce(F.col("capability"), F.array())) > 0,
                    safe_array_join("capability", "; ")
                ).otherwise(F.lit("None listed"))
            ),
            
            # Resources
            F.concat(
                F.lit("Resources: "),
                F.when(F.col("numberDoctors").isNotNull(),
                    F.concat(F.col("numberDoctors").cast("string"), F.lit(" doctors, "))
                ).otherwise(F.lit("")),
                F.when(F.col("capacity").isNotNull(),
                    F.concat(F.col("capacity").cast("string"), F.lit(" beds"))
                ).otherwise(F.lit("No capacity data"))
            ),
            
            # Key capabilities summary
            F.concat(
                F.lit("Key Services: "),
                F.when(F.col("has_emergency"), F.lit("Emergency Care, ")).otherwise(F.lit("")),
                F.when(F.col("has_surgery"), F.lit("Surgical Services, ")).otherwise(F.lit("")),
                F.when(F.col("has_imaging"), F.lit("Medical Imaging, ")).otherwise(F.lit("")),
                F.lit("Capability Score: "),
                F.col("capability_score").cast("string"),
                F.lit("/8")
            ),
            
            # Description (if available)
            F.when(
                F.col("description").isNotNull() & (F.trim(F.col("description")) != ""),
                F.concat(F.lit("About: "), F.col("description"))
            ).when(
                F.col("organizationDescription").isNotNull() & (F.trim(F.col("organizationDescription")) != ""),
                F.concat(F.lit("About: "), F.col("organizationDescription"))
            ).otherwise(F.lit(""))
        )
    )
    
    # Add metadata
    .withColumn("created_at", F.current_timestamp())
    .withColumn(
        "citation_source",
        F.lit("vf_health.ghana.silver_facilities_clean")
    )
)

# Write to Gold
facility_cards.write.mode("overwrite").format("delta").saveAsTable(GOLD_FACILITY_CARDS)

final_cards_count = spark.table(GOLD_FACILITY_CARDS).count()

print(f"\n✅ Gold facility cards created: {GOLD_FACILITY_CARDS}")
print(f"   Total facility cards: {final_cards_count}")

# Show sample of full_text_for_rag
print("\n📝 SAMPLE RAG TEXT (first 500 chars):")
spark.table(GOLD_FACILITY_CARDS).select(
    "name",
    F.substring("full_text_for_rag", 1, 500).alias("rag_text_preview")
).show(3, truncate=False)

print("\n📊 RAG TEXT STATISTICS:")
spark.table(GOLD_FACILITY_CARDS).select(
    F.min(F.length("full_text_for_rag")).alias("min_length"),
    F.avg(F.length("full_text_for_rag")).cast("int").alias("avg_length"),
    F.max(F.length("full_text_for_rag")).alias("max_length")
).show()

print("="*70)
print("✓ Both Gold tables created successfully!")
print(f"  - {GOLD_REGION_SUMMARY}: {spark.table(GOLD_REGION_SUMMARY).count()} regions")
print(f"  - {GOLD_FACILITY_CARDS}: {final_cards_count} facilities")
print("="*70)

In [0]:
SELECT *
FROM vf_health.ghana.gold_medical_deserts
ORDER BY desert_score DESC
LIMIT 20

In [0]:
-- ============================================================================
-- FACILITY PROFILES WITH COMPUTED INTELLIGENCE
-- ============================================================================
-- Shows facilities with their computed intelligence fields

SELECT 
  name,
  organization_type,
  has_ICU,
  has_surgery,
  emergency_ready,
  doctor_ratio,
  ROUND(facility_score, 2) as facility_score
FROM vf_health.ghana.gold_facility_profiles
ORDER BY facility_score DESC
LIMIT 20

In [0]:
-- ============================================================================
-- CRITICAL RISK SIGNALS
-- ============================================================================
-- Shows facilities with critical safety/quality issues

SELECT 
  p.name,
  r.signal_type,
  r.signal_score,
  r.reason,
  r.created_at
FROM vf_health.ghana.gold_risk_signals r
JOIN vf_health.ghana.gold_facility_profiles p ON r.row_id = p.row_id
WHERE r.signal_score >= 0.80  -- CRITICAL/HIGH RISK only
ORDER BY r.signal_score DESC, p.name
LIMIT 30

In [0]:
%python
# ============================================================================
# GOLD SPECIALTY ENRICHMENT
# ============================================================================
# Create enriched specialty table with medical taxonomy classification

from pyspark.sql import functions as F

GOLD_SPECIALTIES = f"{CATALOG}.{SCHEMA}.gold_medical_specialties"

if HAS_SPECIALTY_MODELS:
    print("Creating enriched medical specialties gold table...")
    
    # Explode specialties from silver table
    specialties = (
        df
        .select("row_id", "unique_id", "name", "organization_type", 
                F.explode_outer("specialties").alias("specialty"))
        .filter(F.col("specialty").isNotNull() & (F.trim("specialty") != ""))
    )
    
    # Categorize specialties into high-level groups
    specialties_enriched = (
        specialties
        .withColumn(
            "specialty_category",
            F.when(
                F.lower(F.col("specialty")).rlike(
                    "surgery|surgical|orthopedic|cardiac.*surgery|plastic|neurosurgery"
                ),
                F.lit("surgical")
            ).when(
                F.lower(F.col("specialty")).rlike(
                    "pediatric|obstetric|gynecology|maternity|neonatal"
                ),
                F.lit("maternal_child")
            ).when(
                F.lower(F.col("specialty")).rlike(
                    "emergency|critical.*care|trauma|intensive"
                ),
                F.lit("emergency_critical")
            ).when(
                F.lower(F.col("specialty")).rlike(
                    "radiology|pathology|laboratory|diagnostic"
                ),
                F.lit("diagnostic")
            ).when(
                F.lower(F.col("specialty")).rlike(
                    "cardiology|neurology|nephrology|endocrin|gastro|pulmonary"
                ),
                F.lit("internal_medicine_subspecialty")
            ).when(
                F.lower(F.col("specialty")).rlike(
                    "dental|orthodontic|oral"
                ),
                F.lit("dental")
            ).when(
                F.lower(F.col("specialty")).rlike(
                    "ophthalm|eye|vision"
                ),
                F.lit("ophthalmology")
            ).when(
                F.lower(F.col("specialty")).rlike(
                    "dermatology|skin"
                ),
                F.lit("dermatology")
            ).otherwise(F.lit("general_medical"))
        )
        .withColumn(
            "is_critical_specialty",
            F.col("specialty_category").isin("emergency_critical", "surgical", "maternal_child")
        )
        .withColumn(
            "specialty_id",
            F.sha2(F.concat_ws("||", "row_id", "specialty"), 256)
        )
        .withColumn("created_at", F.current_timestamp())
        .select(
            "specialty_id", "row_id", "unique_id", "name",
            "specialty", "specialty_category", "is_critical_specialty",
            "created_at"
        )
    )
    
    # Write to gold table
    specialties_enriched.write.mode("overwrite").format("delta").saveAsTable(GOLD_SPECIALTIES)
    
    specialty_count = spark.table(GOLD_SPECIALTIES).count()
    print(f"✓ Gold specialties table created: {GOLD_SPECIALTIES}")
    print(f"  Total specialty records: {specialty_count}")
    
    # Show category distribution
    print("\nSpecialty Category Distribution:")
    display(
        spark.table(GOLD_SPECIALTIES)
        .groupBy("specialty_category", "is_critical_specialty")
        .agg(
            F.count("*").alias("count"),
            F.countDistinct("row_id").alias("facilities")
        )
        .orderBy(F.desc("count"))
    )
    
    # Show most common specialties
    print("\nTop 20 Most Common Specialties:")
    display(
        spark.table(GOLD_SPECIALTIES)
        .groupBy("specialty", "specialty_category")
        .agg(
            F.count("*").alias("facility_count")
        )
        .orderBy(F.desc("facility_count"))
        .limit(20)
    )
    
else:
    print("⚠️  Specialty models not available - skipping gold specialty enrichment")
    print("   Standard gold tables will still be created")